In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

RESULTS_DIR = Path("../results/")
DATA_DIR = Path("../dataset_csv/processed/")
FIGURES_DIR = RESULTS_DIR / "final_figures"
FIGURES_DIR.mkdir(exist_ok=True)

results = {}
datasets = ['BLCA', 'BRCA', 'HNSC', 'LUAD', 'UCEC']
for dataset in datasets:
    summary_path = RESULTS_DIR / dataset / "summary.json"
    if summary_path.exists():
        with open(summary_path, 'r') as f:
            results[dataset] = json.load(f)
    else:
        print(f"Warning: Results for {dataset} not found.")

print("Loaded results for:", list(results.keys()))

In [ ]:
print("--- Average Performance Summary ---")
summary_data = []
for dataset, res in results.items():
    summary_data.append({
        "Dataset": dataset,
        "C-index": f"{res['mean_c_index']:.3f} ± {res['std_c_index']:.3f}",
        "Brier Score": f"{res['mean_brier_score']:.3f} ± {res['std_brier_score']:.3f}"
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df)

In [ ]:
for dataset in datasets:
    print(f"--- Generating KM Plot for {dataset} ---")
    
    data_path = DATA_DIR / f"{dataset}_processed_with_folds.csv"
    df = pd.read_csv(data_path)
    
    df['risk_score'] = df['tumor_grade'] 
    median_risk = df['risk_score'].median()
    df['risk_group'] = np.where(df['risk_score'] > median_risk, 'High-Risk', 'Low-Risk')

    plt.figure(figsize=(8, 6))
    ax = plt.subplot(111)

    kmf = KaplanMeierFitter()

    for name, grouped_df in df.groupby('risk_group'):
        kmf.fit(grouped_df['duration'], grouped_df['event'], label=name)
        kmf.plot_survival_function(ax=ax, ci_show=True)

    low_risk = df[df['risk_group'] == 'Low-Risk']
    high_risk = df[df['risk_group'] == 'High-Risk']
    results = logrank_test(low_risk['duration'], high_risk['duration'], low_risk['event'], high_risk['event'])
    
    plt.title(f"{dataset} (P-value: {results.p_value:.2e})")
    plt.xlabel("Timeline (Months)")
    plt.ylabel("Survival Probability")
    plt.grid(True)